# 04 - Speech AI

**AI-900 Domain:** Describe features of conversational AI workloads on Azure

## What You'll Learn
- **Text-to-Speech (TTS)** - Convert text into natural-sounding audio
- **Speech-to-Text (STT)** - Transcribe spoken audio into text
- How Azure Speech Service works

> **Just click `Run All` - no coding needed!**

Note: Audio playback works in Jupyter Notebook (browser). If audio doesn't play, the files are still saved to the `assets/audio/` folder.

## Setup: Load Credentials

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

SPEECH_KEY = os.environ.get("AZURE_SPEECH_KEY", "")
SPEECH_REGION = os.environ.get("AZURE_SPEECH_REGION", "eastus")

if not SPEECH_KEY:
    raise ValueError("AZURE_SPEECH_KEY is not set! Please check your .env file.")

print(f"Credentials loaded! Region: {SPEECH_REGION}")

---
## Part 1: Text-to-Speech (TTS)

Text-to-Speech converts written text into natural-sounding speech.

Azure offers:
- **Standard voices** - Good quality, low cost
- **Neural voices** - Very natural-sounding, powered by deep learning
- **150+ voices** across **60+ languages**

Let's generate some speech!

In [ ]:
import requests
from IPython.display import Audio, display

def text_to_speech(text, voice="en-US-JennyNeural", filename="output.wav"):
    """Convert text to speech using Azure Speech REST API."""
    
    # Get an access token
    token_url = f"https://{SPEECH_REGION}.api.cognitive.microsoft.com/sts/v1.0/issueToken"
    token_resp = requests.post(
        token_url,
        headers={"Ocp-Apim-Subscription-Key": SPEECH_KEY},
        timeout=10
    )
    
    if token_resp.status_code != 200:
        print(f"Failed to get token: {token_resp.status_code}")
        return None
    
    access_token = token_resp.text
    
    # Synthesize speech using SSML
    ssml = f"""<speak version='1.0' xmlns='http://www.w3.org/2001/10/synthesis' xml:lang='en-US'>
        <voice name='{voice}'>{text}</voice>
    </speak>"""
    
    tts_url = f"https://{SPEECH_REGION}.tts.speech.microsoft.com/cognitiveservices/v1"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "riff-24khz-16bit-mono-pcm",
        "User-Agent": "AI900-Demo"
    }
    
    response = requests.post(tts_url, headers=headers, data=ssml.encode("utf-8"), timeout=30)
    
    if response.status_code == 200:
        filepath = os.path.join("..", "assets", "audio", filename)
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"Audio saved to: assets/audio/{filename}")
        return filepath
    else:
        print(f"TTS failed with status {response.status_code}: {response.text[:200]}")
        return None

print("Text-to-Speech function ready!")

### Generate Speech - English

In [ ]:
english_text = "Welcome to the AI 900 hands-on lab! Today you are learning about Azure AI services, including speech recognition and synthesis. This audio was generated by Azure Neural Text to Speech."

print(f"Converting to speech: \"{english_text}\"\n")
audio_path = text_to_speech(english_text, voice="en-US-JennyNeural", filename="tts_english.wav")

if audio_path:
    print("\nPlaying audio (if supported in this environment):")
    display(Audio(audio_path))

### Generate Speech - Different Voice

In [ ]:
text2 = "Azure Speech Service supports many different voices and languages. This is a male voice speaking the same technology description."

print(f"Converting with a different voice: \"{text2}\"\n")
audio_path2 = text_to_speech(text2, voice="en-US-GuyNeural", filename="tts_male_voice.wav")

if audio_path2:
    display(Audio(audio_path2))

### Generate Speech - Another Language (French)

In [ ]:
french_text = "Bonjour! Bienvenue dans le laboratoire Azure AI. La synthese vocale fonctionne dans plus de 60 langues."

print(f"Converting French text: \"{french_text}\"\n")
audio_path3 = text_to_speech(french_text, voice="fr-FR-DeniseNeural", filename="tts_french.wav")

if audio_path3:
    display(Audio(audio_path3))

---
## Part 2: Speech-to-Text (STT)

Speech-to-Text transcribes audio into text. Let's transcribe the audio we just generated!

We'll use the Azure Speech REST API to send audio and get back a transcription.

In [ ]:
def speech_to_text(audio_file_path):
    """Transcribe an audio file using Azure Speech REST API."""
    
    stt_url = f"https://{SPEECH_REGION}.stt.speech.microsoft.com/speech/recognition/conversation/cognitiveservices/v1?language=en-US"
    
    headers = {
        "Ocp-Apim-Subscription-Key": SPEECH_KEY,
        "Content-Type": "audio/wav; codecs=audio/pcm; samplerate=24000"
    }
    
    with open(audio_file_path, "rb") as audio_file:
        response = requests.post(
            stt_url,
            headers=headers,
            data=audio_file,
            timeout=30
        )
    
    if response.status_code == 200:
        result = response.json()
        if result.get("RecognitionStatus") == "Success":
            return result.get("DisplayText", "")
        else:
            return f"Recognition status: {result.get('RecognitionStatus')}"
    else:
        return f"STT failed with status {response.status_code}: {response.text[:200]}"

print("Speech-to-Text function ready!")

### Transcribe the Generated Audio

Let's transcribe the English audio we generated in Part 1 and compare it with the original text.

In [ ]:
english_audio = os.path.join("..", "assets", "audio", "tts_english.wav")

if os.path.exists(english_audio):
    print("Transcribing audio...\n")
    transcription = speech_to_text(english_audio)
    
    print("=" * 60)
    print("SPEECH-TO-TEXT RESULTS")
    print("=" * 60)
    print(f"\nOriginal text:")
    print(f"  \"{english_text}\"")
    print(f"\nTranscribed text:")
    print(f"  \"{transcription}\"")
    print(f"\nMatch: The transcription {'closely matches' if len(transcription) > 20 else 'may differ from'} the original!")
else:
    print("Audio file not found. Please run the TTS cells above first.")

---
## Key Concepts for AI-900

| Feature | Direction | Description |
|---------|-----------|-------------|
| **Text-to-Speech (TTS)** | Text -> Audio | Converts written text to spoken audio |
| **Speech-to-Text (STT)** | Audio -> Text | Transcribes spoken audio to written text |
| **Neural Voices** | - | Deep learning-based voices that sound very natural |
| **SSML** | - | Speech Synthesis Markup Language for controlling voice, speed, pitch |

### AI-900 Exam Tips
- **Azure Speech Service** is part of Azure AI Services
- TTS and STT are separate features of the same service
- **Neural voices** produce the most natural-sounding speech
- **SSML** (Speech Synthesis Markup Language) lets you control pronunciation, speed, and pitch
- Speech-to-Text supports **real-time** and **batch** transcription
- Over **60 languages** are supported for both TTS and STT
- **Custom Speech** lets you train models for specific accents or terminology

---
**Next:** Open `05_generative_ai.ipynb` to explore GPT models and prompt engineering!